In [1]:
import pymupdf
import spacy
import re
import pandas as pd
import numpy as np
import unicodedata
import os
from pathlib import Path

In [5]:
nlp = spacy.load('en_core_web_lg')

In [44]:
# word vectors (similarity) vs linguistic featues e.g. lemma e.g. transportation vs transport

all_cat_words = ['renewable', 'solar', 'wind', 'bioenergy', 'biofuel', 'biomass', 'hydropower', 'hydrogen',
                 'power', 'grid', 'transmission', 'generation', 'efficiency', 'retrofit', 'pollution', 'waste',
                 'land', 'agriculture', 'forestry', 'forest', 'fisheries', 'food', 'terrestrial', 'aquatic',
                   'biodiversity', 'conservation', 'transportation', 'electric', 'battery', 'EV', 'charger',
                     'bus', 'rail', 'train', 'car', 'vehicle', 'bicycle', 'non-motorized', 'aviation',
                     'water', 'potable', 'wastewater', 'sanitation', 'treatment', 'adaptation', 'disaster',
                     'circular', 'recycle', 'reuse','buildings', 'appliances', 'housing']

renewable_energy = ['renewable', 'solar', 'wind', 'bioenergy', 'biofuel', 'biomass', 'hydropower', 'hydrogen', 'power', 'grid', 'transmission', 'generation']
energy_efficiency = ['efficiency', 'retrofit']
pollution_prevention_and_control = ['pollution', 'waste']
environmentally_sustainable_management_of_living_natural_resources_and_land_use = ['land', 'agriculture', 'forestry', 'forest', 'fisheries', 'food']
terrestrial_and_aquatic_biodiversity_conservation = ['terrestrial', 'aquatic', 'biodiversity', 'conservation']
clean_transportation = ['transportation', 'electric', 'battery', 'EV', 'charger', 'bus', 'rail', 'train', 'car', 'vehicle', 'bicycle', 'non-motorized', 'aviation']
sustainable_water_and_wastewater_management = ['water', 'potable', 'wastewater', 'sanitation', 'treatment']
climate_change_adaptation = ['adaptation', 'disaster']
circular_economy_and_or_ecoefficient_projects = ['circular', 'recycle', 'reuse']
green_buildings = ['buildings', 'appliances', 'housing']

In [94]:
def catTable(document):
    
    pdf = pymupdf.open(document) 

    tableCount = 0
    tablePages = []
    tableHeaders = []
    tableDFs = []


    for page_idx in range(len(pdf)): 
        page = pdf[page_idx]

        find_tables = page.find_tables(strategy = 'lines_strict')
        if find_tables.tables:
            table_count = len(find_tables.tables)
            tableheader = find_tables[0].header.names # for now assume only 1 table per page
            tableDF = find_tables[0].to_pandas()
            tableCount += table_count
            tablePages.append(page_idx)
            tableHeaders.append(tableheader)
            tableDFs.append(tableDF)

    return tableCount, tablePages, tableHeaders, tableDFs

        

In [16]:
def tableAssess(language, tableCount, tablePages, tableHeaders, tableDFs):

    if language == 'EN':
        keywordsCAT = ['Category', 'Categories', 'Criteria', 'Criterion', 'Green Sectors']

    tablePageCurrent = 0
    tablePagePrior = 0
    uniqueCats = []
    uniqueCatsNoDupes = []

    if tableCount == 1: # applicable if only 1 table detected in whole pdf
        dataFrame = tableDFs[0]
        header = tableHeaders[0]
        for keyword in keywordsCAT:
            if keyword.lower() in header[0].lower(): # assumes category col is always the first (LHS) col
                colName = header[0]
                uniqueC = dataFrame[colName].dropna().unique().tolist()
                for messyCat in uniqueC:
                    if messyCat:
                        tidyCat = messyCat.replace('\n',' ').lower()
                        uniqueCatsNoDupes.append(tidyCat)
    elif tableCount > 1:
        for idx in range(len(tableDFs)): # count of tableHeaders and DFs should be the same, based on the page level find_tables object
            tablePageCurrent = tablePages[idx]
            dataFrame = tableDFs[idx]
            header = tableHeaders[idx]
            for keyword in keywordsCAT:
                if keyword.lower() in header[0].lower():
                    colName = header[0]
                    uniqueC = dataFrame[colName].dropna().unique().tolist()
                    for messyCat in uniqueC:
                        if messyCat:
                            tidyCat = messyCat.replace('\n',' ').lower()
                            uniqueCats.append(tidyCat)
                elif header[0] == 'Col0' and tablePageCurrent - tablePagePrior == 1:
                    uniqueC = dataFrame['Col0'].dropna().unique().tolist()
                    if uniqueC:
                        for messyCat in uniqueC:
                            if messyCat:
                                tidyCat = messyCat.replace('\n',' ').lower()
                                uniqueCats.append(tidyCat)
            tablePagePrior = tablePageCurrent
        uniqueCatsNoDupes = list(set(uniqueCats))

    return uniqueCatsNoDupes

# good example of use of AI: Gemini: https://www.google.com/search?client=safari&rls=en&q=I+just+want+the+unique+categories+of+a+dataframe+column+returned+by+unique%28%29+so+how+do+I+discard+the+other+info+that+is+returned%3F+the+unique+values+are+in+a+list+like+structure+inside+the+numpy+array&ie=UTF-8&oe=UTF-8
# 
        

In [10]:
def fontInfo(language, document):
    
    if language == 'EN':
        keywordsUOP = ['Use of Proceeds', 'Use of Funds', 'Use of the Proceeds']
        keywordsSEEGP = ['Selection and Evaluation', 'Process for the Project Evaluation and Selection', 'Project Evaluation and Selection Process', 'Project Selection and Evaluation Process', 'Process for Project Evaluation and Selection', 'Project Selection and Assessment Process', 'Project Selection Criteria', 'Project evaluation & selection', 'Project Selection Process']
        keywordsSEEGP_test = ['Selection and Evaluation', 'Process for', 'Evaluation and Selection', 'Project Evaluation', 'Assessment Process', 'Selection process', 'Project selection criteria']


    results_UOP = []
    results_SEEGP = []
    
    pdf = pymupdf.open(document) 

    for page_idx in range(len(pdf)): 
        page = pdf[page_idx]
        dict = page.get_text("dict")
        blocks = dict["blocks"] 
        for block in blocks:
            if "lines" in block.keys():
                spans = block['lines']
                for span in spans:
                    data = span['spans']
                    for lines in data:
                        for keyword in keywordsUOP:
                            if keyword.lower() in lines['text'].lower().strip():
                                results_UOP.append((lines['text'], lines['size'], lines['bbox'], page_idx))
                        for keyword in keywordsSEEGP_test:
                            if keyword.lower() in lines['text'].lower().strip():
                                results_SEEGP.append((lines['text'], lines['size'], lines['bbox'], page_idx))

                            

    return results_UOP, results_SEEGP

In [11]:
def find_max_font(results_UOP, results_SEEGP):
    
    max_font_size = 0
    for result_idx in range(len(results_UOP)):
        result = results_UOP[result_idx]
        if result[1] > max_font_size:
            max_font_size = result[1]
            max_idx_UOP = result_idx

    max_font_size = 0
    if len(results_SEEGP) == 0:
        max_idx_SEEGP = 0
    else:       
        for result_idx in range(len(results_SEEGP)):
            result = results_SEEGP[result_idx]
            if result[1] > max_font_size:
                max_font_size = result[1]
                max_idx_SEEGP = result_idx

    return max_idx_UOP, max_idx_SEEGP

In [12]:
def tocCheck(results_UOP, results_SEEGP, max_idx_UOP, max_idx_SEEGP):
    
    if len(results_SEEGP) == 0:
        areaUOP = results_UOP[max_idx_UOP]
        areaUOP_page = areaUOP[3]
        areaUOP_coords = areaUOP[2]
        areaUOP_y1 = areaUOP_coords[3]
        areaSEEGP_page = 0
        areaSEEGP_y0 = 0

    else:
        areaUOP = results_UOP[max_idx_UOP]
        areaSEEGP = results_SEEGP[max_idx_SEEGP]
        areaUOP_page = areaUOP[3]
        areaSEEGP_page = areaSEEGP[3]
        areaUOP_coords = areaUOP[2]
        areaSEEGP_coords = areaSEEGP[2]
        areaUOP_y1 = areaUOP_coords[3]
        areaSEEGP_y0 = areaSEEGP_coords[1]

        if areaUOP_page == areaSEEGP_page:
            if 0 < areaSEEGP_y0 - areaUOP_y1 < 40:
                results_UOP_remove = results_UOP.pop(max_idx_UOP)
                results_SEEGP_remove = results_SEEGP.pop(max_idx_SEEGP)

                max_idx_UOP, max_idx_SEEGP = find_max_font(results_UOP, results_SEEGP)

                areaUOP = results_UOP[max_idx_UOP]
                areaSEEGP = results_SEEGP[max_idx_SEEGP]
                areaUOP_page = areaUOP[3]
                areaSEEGP_page = areaSEEGP[3]
                areaUOP_coords = areaUOP[2]
                areaSEEGP_coords = areaSEEGP[2]
                areaUOP_y1 = areaUOP_coords[3]
                areaSEEGP_y0 = areaSEEGP_coords[1]

    return areaUOP_page, areaUOP_y1, areaSEEGP_page, areaSEEGP_y0

In [13]:
def keepSEEGP(areaUOP_page, areaUOP_y1, areaSEEGP_page, areaSEEGP_y0):

    if areaSEEGP_page == areaUOP_page:
        if areaSEEGP_y0 < areaUOP_y1:
            keep_areaSEEGP = False
        elif areaSEEGP_y0 > areaUOP_y1:
            keep_areaSEEGP = True
    elif areaSEEGP_page > areaUOP_page:
        keep_areaSEEGP = True
    elif areaSEEGP_page < areaUOP_page:
        keep_areaSEEGP = False

    return keep_areaSEEGP

In [26]:
# page scenarios and extract words when keep_areaSEEGP = True

def wordsUOP_endPage(document, areaUOP_page, areaUOP_y1, areaSEEGP_page, areaSEEGP_y0):

    pdf = pymupdf.open(document)
    extractUOPwords = []

    # iterate pages
    for page_idx in range(len(pdf)):
        page = pdf[page_idx]

       
# process four page scenarios and extract text as words
    # A: UOP all on a single page
        if page_idx == areaUOP_page and page_idx == areaSEEGP_page:
            for word in page.get_text('words'):
                x0, y0, x1, y1, messyText, *_ = word
                if y0 > areaUOP_y1 and y1 < areaSEEGP_y0:
                    tidyText = messyText.replace('\n',' ').lower()
                    extractUOPwords.append(tidyText)

    # UOP across >1 page
    # B: current page is start page
        elif page_idx == areaUOP_page:
            for word in page.get_text('words'):
                x0, y0, x1, y1, messyText, *_ = word
                if y0 > areaUOP_y1:
                    tidyText = messyText.replace('\n',' ').lower()
                    extractUOPwords.append(tidyText)

    # D: current page is neither start nor end page but is in the UOP area
        elif page_idx > areaUOP_page and page_idx < areaSEEGP_page:
            for word in page.get_text('words'):
                x0, y0, x1, y1, messyText, *_ = word
                tidyText = messyText.replace('\n',' ').lower()
                extractUOPwords.append(tidyText)

    # C: current page is end page
        elif page_idx == areaSEEGP_page:
                for word in page.get_text('words'):
                    x0, y0, x1, y1, messyText, *_ = word
                    if y1 < areaSEEGP_y0:
                        tidyText = messyText.replace('\n',' ').lower()
                        extractUOPwords.append(tidyText)

    return extractUOPwords



In [36]:
# iterate pages in the pdf and extract words, preferrably between the UOP and SEEGP sections or else from UOP to end

def wordsUOP(keep_areaSEEGP, document, areaUOP_page, areaUOP_y1, areaSEEGP_page, areaSEEGP_y0):
    # language = language

    pdf = pymupdf.open(document)

    if keep_areaSEEGP == True:
        extractUOPwords = wordsUOP_endPage(document, areaUOP_page, areaUOP_y1, areaSEEGP_page, areaSEEGP_y0)
        UOPwordsNoDupes = list(set(extractUOPwords))

    elif keep_areaSEEGP == False:
        extractUOPwords = []

# iterate pages
        for page_idx in range(len(pdf)):
            page = pdf[page_idx]
       
    # process two page scenarios and extract text as words

        # assume UOP across > 1 page
        # B: current page is start page
            if page_idx == areaUOP_page:
                for word in page.get_text('words'):
                    x0, y0, x1, y1, messyText, *_ = word
                    if y0 > areaUOP_y1:
                        tidyText = messyText.replace('\n',' ').lower()
                        extractUOPwords.append(tidyText)

        # E: current page is after the UOP start page
            elif page_idx > areaUOP_page:
                for word in page.get_text('words'):
                    x0, y0, x1, y1, messyText, *_ = word
                    tidyText = messyText.replace('\n',' ').lower()
                    extractUOPwords.append(tidyText)
    
        UOPwordsNoDupes = list(set(extractUOPwords))
    
    return UOPwordsNoDupes


In [59]:
# process words into category word dataframes where token.similarity score passes threshold

def UOPwords_to_Catwords(UOPwordsNoDupes):

    all = all_cat_words
    re = renewable_energy
    ee = energy_efficiency
    ppc = pollution_prevention_and_control
    esml = environmentally_sustainable_management_of_living_natural_resources_and_land_use
    tabc = terrestrial_and_aquatic_biodiversity_conservation
    ct = clean_transportation
    swwm = sustainable_water_and_wastewater_management
    cca = climate_change_adaptation
    ce = circular_economy_and_or_ecoefficient_projects
    gb = green_buildings

    similarity_threshold = 0.72

    # outputs
    simsre = {}
    simsee = {}
    simsppc = {}
    simsesml = {}
    simstabc = {}
    simsct = {}
    simsswwm = {}
    simscca = {}
    simsce = {}
    simsgb = {}
    

    # renewable_energy
    for worda in re:
        doca = nlp(worda)
        for wordb in UOPwordsNoDupes:
            docb = nlp(wordb)
            if doca.similarity(docb) >= similarity_threshold:
                sim = doca.similarity(docb)
                simsre[doca[0].text + ' ' + docb[0].text] = sim

    # energy_efficiency
    for wordc in ee:
        docc = nlp(wordc)
        for wordb in UOPwordsNoDupes:
            docb = nlp(wordb)
            if docc.similarity(docb) >= similarity_threshold:
                sim = docc.similarity(docb)
                simsee[docc[0].text + ' ' + docb[0].text] = sim
    
    # pollution_prevention_and_control
    for wordd in ppc:
        docd = nlp(wordd)
        for wordb in UOPwordsNoDupes:
            docb = nlp(wordb)
            if docd.similarity(docb) >= similarity_threshold:
                sim = docd.similarity(docb)
                simsppc[docd[0].text + ' ' + docb[0].text] = sim

    # environmentally_sustainable_management_of_living_natural_resources_and_land_use
    for worde in esml:
        doce = nlp(worde)
        for wordb in UOPwordsNoDupes:
            docb = nlp(wordb)
            if doce.similarity(docb) >= similarity_threshold:
                sim = doce.similarity(docb)
                simsesml[doce[0].text + ' ' + docb[0].text] = sim

    # terrestrial_and_aquatic_biodiversity_conservation
    for wordf in tabc:
        docf = nlp(wordf)
        for wordb in UOPwordsNoDupes:
            docb = nlp(wordb)
            if docf.similarity(docb) >= similarity_threshold:
                sim = docf.similarity(docb)
                simstabc[docf[0].text + ' ' + docb[0].text] = sim

    # clean_transportation
    for wordg in ct:
        docg = nlp(wordg)
        for wordb in UOPwordsNoDupes:
            docb = nlp(wordb)
            if docg.similarity(docb) >= similarity_threshold:
                sim = docg.similarity(docb)
                simsct[docg[0].text + ' ' + docb[0].text] = sim

    # sustainable_water_and_wastewater_management
    for wordh in swwm:
        doch = nlp(wordh)
        for wordb in UOPwordsNoDupes:
            docb = nlp(wordb)
            if doch.similarity(docb) >= similarity_threshold:
                sim = doch.similarity(docb)
                simsswwm[doch[0].text + ' ' + docb[0].text] = sim

    # climate_change_adaptation
    for wordi in cca:
        doci = nlp(wordi)
        for wordb in UOPwordsNoDupes:
            docb = nlp(wordb)
            if doci.similarity(docb) >= similarity_threshold:
                sim = doci.similarity(docb)
                simscca[doci[0].text + ' ' + docb[0].text] = sim

        # circular_economy_and_or_ecoefficient_projects
    for wordj in ce:
        docj = nlp(wordj)
        for wordb in UOPwordsNoDupes:
            docb = nlp(wordb)
            if docj.similarity(docb) >= similarity_threshold:
                sim = docj.similarity(docb)
                simsce[docj[0].text + ' ' + docb[0].text] = sim

    # green_buildings
    for wordk in gb:
        dock = nlp(wordk)
        for wordb in UOPwordsNoDupes:
            docb = nlp(wordb)
            if dock.similarity(docb) >= similarity_threshold:
                sim = dock.similarity(docb)
                simsgb[dock[0].text + ' ' + docb[0].text] = sim

    return simsre, simsee, simsppc, simsesml, simstabc, simsct, simsswwm, simscca, simsce, simsgb

In [65]:
def finaliseUniqueCats(uniqueCatsNoDupes):

    all = all_cat_words
    CatwordsFinal = []

    cleanup_TH = 0.72

    if len(uniqueCatsNoDupes) > 0: 
    # all cat words - clean up spurious elements from table extraction
    # all cat words - remove excess UOP word extracts
        for wordx in all:
            docx = nlp(wordx)
            for cat in uniqueCatsNoDupes:
                docCat = nlp(cat)
                if docx.similarity(docCat) >= cleanup_TH:
                    CatwordsFinal.append(cat)
                    CatwordsFFinal = list(set(CatwordsFinal))

    return CatwordsFFinal




        #for wordy in UOPwordsNoDupes:
            #docy = nlp(wordy)
            #if docx.similarity(docy) <= cleanup_TH:
                #UOPwordsFinal = UOPwordsNoDupes.remove(wordy)


In [101]:
# run the program


document = '/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/refinement_reports/Green_Bond_Financing_Framework.pdf'
language = 'EN'

filepath = Path(document)

tableCount, tablePages, tableHeaders, tableDFs = catTable(document)
uniqueCatsNoDupes = tableAssess(language, tableCount, tablePages, tableHeaders, tableDFs)

results_UOP, results_SEEGP = fontInfo(language, document)
max_idx_UOP, max_idx_SEEGP = find_max_font(results_UOP, results_SEEGP)
areaUOP_page, areaUOP_y1, areaSEEGP_page, areaSEEGP_y0 = tocCheck(results_UOP, results_SEEGP, max_idx_UOP, max_idx_SEEGP)
keep_areaSEEGP = keepSEEGP(areaUOP_page, areaUOP_y1, areaSEEGP_page, areaSEEGP_y0)
UOPwordsNoDupes = wordsUOP(keep_areaSEEGP, document, areaUOP_page, areaUOP_y1, areaSEEGP_page, areaSEEGP_y0)

if len(uniqueCatsNoDupes) == 0:
    simsre, simsee, simsppc, simsesml, simstabc, simsct, simsswwm, simscca, simsce, simsgb = UOPwords_to_Catwords(UOPwordsNoDupes)
    print('')
    print(areaUOP_page, areaUOP_y1)
    print(keep_areaSEEGP)
    if keep_areaSEEGP == True:
        print(areaSEEGP_page, areaSEEGP_y0)
    print(len(UOPwordsNoDupes))
    print('')
    print(f"renewable_energy  {simsre}")
    print(f"energy_efficiency {simsee}")
    print(f"pollution_prevention_and_control {simsppc}")
    print(f"environmentally_sustainable_management_of_living_natural_resources_and_land_use {simsesml}")
    print(f"terrestrial_and_aquatic_biodiversity_conservation {simstabc}")
    print(f"clean_transportation {simsct}")
    print(f"sustainable_water_and_wastewater_management {simsswwm}")
    print(f"climate_change_adaptation {simscca}")
    print(f"circular_economy_and_or_ecoefficient_projects {simsce}")
    print(f"green_buildings {simsgb}")
else:
    print('categories from tables')
    print(tableCount)
    print(tablePages)
    print(uniqueCatsNoDupes)
    CatwordsFinal = finaliseUniqueCats(uniqueCatsNoDupes)
    print(CatwordsFinal)


categories from tables
3
[7, 8, 10]
['clean transportation', 'energy efficiency']
['clean transportation', 'energy efficiency']
